In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window


dbutils.widgets.text("catalogo", "workspace")
catalogo = dbutils.widgets.get("catalogo")

# Cria o schema da Silver
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.silver")


DataFrame[]

In [0]:
# leitura da tabela da Bronze
df = spark.table(f"{catalogo}.bronze.tb_movies_reviews")

# Renomeia para português, tira espaços das pontas e converte a nota para número.
# try_cast: se a nota não for um número válido, vira NULL em vez de dar erro.
df = df.select(
    F.trim(F.col("id")).alias("id_filme"),
    F.trim(F.col("nome")).alias("nome_usuario"),
    F.expr("try_cast(nota AS DOUBLE)").alias("nota_usuario"),
    F.trim(F.col("comentario")).alias("comentario_usuario"),
)

# Regra de negócio, então a nota só vale se for entre 0 e 10. Fora disso vira NULL.
df = df.withColumn(
    "nota_usuario",
    F.when((F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10), F.col("nota_usuario"))
)

# Comentário vazio ou só com espaços vira "Sem comentário".
# Como já foi aplicado trim, só com espaços virou "", então basta testar NULL ou vazio.
df = df.withColumn(
    "comentario_usuario",
    F.when(F.col("comentario_usuario").isNull() | (F.col("comentario_usuario") == ""), "Sem comentário")
     .otherwise(F.col("comentario_usuario"))
)

# Remove as duplicatas
df = df.dropDuplicates()

# Silver é recalculada inteira a partir da Bronze, por isso overwrite
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.silver.tb_avaliacoes_usuarios")

print(df.count())
display(df.limit(10))

32412


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
637007,Lucas Reis 602,3.9,Sem comentário
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,Sem comentário
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


In [0]:
df = spark.table(f"{catalogo}.bronze.tb_movies_metrics")

# Popularidade vem com dois separadores decimais: "72.735" e "154,34".
# Troca vírgula por ponto antes de converter, senão "154,34" viraria NULL.
df = df.withColumn("pop_texto", F.regexp_replace(F.trim(F.col("popularity")), ",", "."))

# Renomeia e converte. Por causa do deslocamento de colunas, há textos nas colunas de nota e votos;
# o try_cast transforma esses textos em NULL sem interromper o notebook.
df = df.select(
    F.trim(F.col("id")).alias("id_filme"),
    F.expr("try_cast(pop_texto AS DOUBLE)").alias("popularidade"),
    F.expr("try_cast(vote_average AS DOUBLE)").alias("nota_media_tmdb"),
    F.expr("try_cast(vote_count AS INT)").alias("qtd_votos_tmdb"),
    F.expr("try_cast(averageRating AS DOUBLE)").alias("nota_media_imdb"),
    F.expr("try_cast(numVotes AS INT)").alias("qtd_votos_imdb"),
    F.col("ingestion_datetime"),
)

# Regras de negócio, faz com que notas fora de 0 a 10 viram NULL 
df = df.withColumn("nota_media_tmdb", F.when((F.col("nota_media_tmdb") >= 0) & (F.col("nota_media_tmdb") <= 10), F.col("nota_media_tmdb")))
df = df.withColumn("nota_media_imdb", F.when((F.col("nota_media_imdb") >= 0) & (F.col("nota_media_imdb") <= 10), F.col("nota_media_imdb")))

# popularidade e contagens de votos negativas viram NULL
df = df.withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
df = df.withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
df = df.withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))

# Deduplicação: o mesmo filme aparece mais de uma vez.
# Fica a linha da carga mais recente. Como as repetidas vieram no mesmo arquivo (mesmo ingestion_datetime),
# o desempate é pela linha com mais campos preenchidos.
df = df.withColumn(
    "preenchidos",
    F.when(F.col("popularidade").isNotNull(), 1).otherwise(0)
    + F.when(F.col("nota_media_tmdb").isNotNull(), 1).otherwise(0)
    + F.when(F.col("qtd_votos_tmdb").isNotNull(), 1).otherwise(0)
    + F.when(F.col("nota_media_imdb").isNotNull(), 1).otherwise(0)
    + F.when(F.col("qtd_votos_imdb").isNotNull(), 1).otherwise(0)
)

janela = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc(), F.col("preenchidos").desc())

df = df.withColumn("ordem", F.row_number().over(janela))
df = df.filter(F.col("ordem") == 1)
df = df.drop("ordem", "preenchidos", "ingestion_datetime")

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.silver.tb_metricas_engajamento")

print(df.count())
display(df.limit(10))

99013


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000004,1.132,0.0,0,6.8,27
1000005,0.6,0.0,0,null,40
1000007,0.6,0.0,0,4.7,10
1000011,1.169,0.0,0,4.8,16
1000014,0.6,0.0,0,7.2,15
1000030,0.615,0.0,0,null,25
1000054,0.6,0.0,0,5.9,10
1000058,1.489,6.75,6,6.2,382
1000059,0.6,0.0,0,7.7,25
1000073,13.212,6.8,15,null,236


In [0]:
df = spark.table(f"{catalogo}.bronze.tb_movies_info")

# STATUS
# Normaliza antes de traduzir: caixa alta, hífen vira espaço e espaços repetidos viram um só.
# Assim "In-Production", "IN PRODUCTION" e "in production" ficam todos "IN PRODUCTION".
status = F.upper(F.trim(F.col("status")))
status = F.regexp_replace(status, "-", " ")
status = F.regexp_replace(status, " +", " ")
df = df.withColumn("status_norm", status)

# Traduz. O que não for um status conhecido (inclusive datas que caíram nessa coluna) vira "Não Informado".
df = df.withColumn(
    "status_filme",
    F.when(F.col("status_norm") == "RELEASED", "Lançado")
     .when(F.col("status_norm") == "POST PRODUCTION", "Pós-Produção")
     .when(F.col("status_norm") == "IN PRODUCTION", "Em Produção")
     .when(F.col("status_norm") == "PLANNED", "Planejado")
     .when(F.col("status_norm") == "RUMORED", "Rumores")
     .when(F.col("status_norm") == "CANCELED", "Cancelado")
     .otherwise("Não Informado")
)

# DATA DE LANÇAMENTO
# A origem tem três formatos: 2016-02-09, 25/04/2018 e 04-25-2018.
# Testa um formato de cada vez; o coalesce pega o primeiro que deu certo.
# Só fica NULL se nenhum formato funcionar.

# Um dealhe importante a ser comentado, é que em datas que é utilizada a /,
# é contado dia/mes/ano, e quando é utilizado - é contado como mes-dia-ano,
# por isso foram invertidas no codigo abaixo. 
df = df.withColumn(
    "data_lancamento",
    F.coalesce(
        F.expr("try_to_timestamp(trim(release_date), 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(trim(release_date), 'dd/MM/yyyy')"),
        F.expr("try_to_timestamp(trim(release_date), 'MM-dd-yyyy')"),
    ).cast("date")
)

# Coluna derivada: ano extraído da data
df = df.withColumn("ano_lancamento", F.year(F.col("data_lancamento")))

# Renomeia e converte as demais colunas
df = df.select(
    F.trim(F.col("id")).alias("id_filme"),
    F.trim(F.col("title")).alias("titulo"),
    F.trim(F.col("original_title")).alias("titulo_original"),
    F.col("data_lancamento"),
    F.col("ano_lancamento"),
    F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"),
    F.trim(F.col("original_language")).alias("idioma_original"),
    F.col("status_filme"),
    F.trim(F.col("overview")).alias("sinopse"),
    F.trim(F.col("tagline")).alias("frase_divulgacao"),
    F.col("ingestion_datetime"),
)

# DEDUPLICAÇÃO: uma linha por filme, a mais recente.
# Desempate dentro do mesmo lote: a linha com mais campos preenchidos
# (há duplicatas em que uma das versões perdeu a sinopse).
df = df.withColumn(
    "preenchidos",
    F.when(F.col("data_lancamento").isNotNull(), 1).otherwise(0)
    + F.when(F.col("duracao_minutos").isNotNull(), 1).otherwise(0)
    + F.when(F.col("sinopse").isNotNull(), 1).otherwise(0)
    + F.when(F.col("frase_divulgacao").isNotNull(), 1).otherwise(0)
)

janela = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc(), F.col("preenchidos").desc())
df = df.withColumn("ordem", F.row_number().over(janela))
df = df.filter(F.col("ordem") == 1)
df = df.drop("ordem", "preenchidos", "ingestion_datetime")

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.silver.tb_info_filmes")

print(df.count())
display(df.groupBy("status_filme").count())

97611


status_filme,count
Lançado,96261
Em Produção,604
Pós-Produção,697
Planejado,47
Não Informado,2


In [0]:
df = spark.table(f"{catalogo}.bronze.tb_cotacao_dolar")

# Converte a data e hora da API para data, e a cotação para decimal com 4 casas (padrão da PTAX)
df = df.select(
    F.to_date(F.expr("try_cast(dataHoraCotacao AS TIMESTAMP)")).alias("data_cotacao"),
    F.col("cotacaoCompra").cast("decimal(10,4)").alias("cotacao_compra"),
    F.col("ingestion_datetime"),
)

# Uma cotação por dia. Se o mesmo dia veio em mais de uma carga, fica a mais recente.
janela_dia = Window.partitionBy("data_cotacao").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("ordem", F.row_number().over(janela_dia))
df = df.filter(F.col("ordem") == 1).drop("ordem", "ingestion_datetime")

# Calendário com todos os dias, do primeiro dia com cotação até hoje (inclui sábados, domingos e feriados)
datas = df.agg(F.min("data_cotacao").alias("inicio"), F.max("data_cotacao").alias("fim")).first()
calendario = spark.sql(f"""
    SELECT explode(sequence(DATE'{datas['inicio']}', greatest(DATE'{datas['fim']}', current_date()), INTERVAL 1 DAY)) AS data_cotacao
""")

# Junta o calendário com as cotações. Dias sem cotação ficam com NULL.
df = calendario.join(df, "data_cotacao", "left")

# Marca quais dias não tinham cotação, antes de preencher
df = df.withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())

# Forward fill: cada dia recebe a última cotação não nula até ele
janela_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df = df.withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(janela_ffill))

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.silver.tb_cotacao_dolar")

display(df.orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra,cotacao_preenchida
2026-09-29,5.2198,false
2026-09-30,5.1803,false
2026-10-01,5.2073,false
2026-10-02,5.2232,false
2026-10-03,5.2232,true
2026-10-04,5.2232,true
2026-10-05,4.9853,false
2026-10-06,4.9692,false


In [0]:
df = spark.table(f"{catalogo}.bronze.tb_movies_financials")

# Função que limpa uma coluna de valor em dinheiro e cria a coluna nova já convertida
def limpar_valor(df, origem, destino):
    valor = F.upper(F.trim(F.col(origem)))

    # Textos que significam "sem dado" viram NULL antes de qualquer conversão
    valor = F.when(valor.isin("UNKNOWN", "NÃO INFORMADO", "N/A", ""), None).otherwise(valor)

    # Remove símbolo de moeda e espaços: "$ 97000000" e "USD 150000000" viram só o número
    valor = F.regexp_replace(valor, r"USD|\$|\s", "")

    # Sufixo M = milhão, K = mil. Sem isso, "34.0M" viraria 34 dólares.
    df = df.withColumn("multiplicador", F.when(valor.endswith("M"), 1000000).when(valor.endswith("K"), 1000).otherwise(1))

    # Tira o sufixo e a vírgula de milhar, deixando só o número
    df = df.withColumn("texto_numero", F.regexp_replace(valor, "[MK,]", ""))

    # Converte para decimal e aplica o multiplicador
    df = df.withColumn(destino, (F.expr("try_cast(texto_numero AS DECIMAL(18,2))") * F.col("multiplicador")).cast("decimal(18,2)"))

    # Regra de negócio: zero ou negativo é falta de dado, vira NULL
    df = df.withColumn(destino, F.when(F.col(destino) > 0, F.col(destino)))

    return df.drop("multiplicador", "texto_numero")


df = limpar_valor(df, "budget", "orcamento_usd")
df = limpar_valor(df, "revenue", "receita_usd")
df = df.withColumn("id_filme", F.trim(F.col("id")))

# Deduplicação: uma linha por filme, a mais recente; desempate pela linha com mais valores preenchidos
df = df.withColumn(
    "preenchidos",
    F.when(F.col("orcamento_usd").isNotNull(), 1).otherwise(0)
    + F.when(F.col("receita_usd").isNotNull(), 1).otherwise(0)
)
janela = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc(), F.col("preenchidos").desc())
df = df.withColumn("ordem", F.row_number().over(janela))
df = df.filter(F.col("ordem") == 1)

# Cotação usada na conversão: a mais recente da Silver.
# A base não tem a cotação da data de cada filme, então todos são convertidos pela última PTAX disponível.
cotacao = (
    spark.table(f"{catalogo}.silver.tb_cotacao_dolar")
    .orderBy(F.col("data_cotacao").desc())
    .first()["cotacao_compra"]
)
print("Cotação usada:", cotacao)

df = df.withColumn("orcamento_brl", (F.col("orcamento_usd") * cotacao).cast("decimal(18,2)"))
df = df.withColumn("receita_brl", (F.col("receita_usd") * cotacao).cast("decimal(18,2)"))

# Lucro só quando orçamento e receita existem. Se um deles é NULL, o lucro fica NULL
# (tratar a falta como zero inventaria lucro ou prejuízo que não existe).
tem_os_dois = F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull()
df = df.withColumn("lucro_usd", F.when(tem_os_dois, F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
df = df.withColumn("lucro_brl", F.when(tem_os_dois, F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))

# Margem = lucro / receita x 100. Só calcula com receita maior que zero, para não dividir por zero.
df = df.withColumn(
    "margem_lucro_pct",
    F.when(F.col("lucro_usd").isNotNull() & (F.col("receita_usd") > 0),
           F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2)).cast("decimal(18,2)")
)

df = df.select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl", "lucro_usd", "lucro_brl", "margem_lucro_pct")

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.silver.tb_financeiro_filmes")

print(df.count())
display(df.filter(F.col("lucro_usd").isNotNull()).limit(10))

Cotação usada: 4.9692
99006


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
1001724,3000.00,8000.00,14907.60,39753.60,5000.00,24846.00,62.50
1003578,10000.00,3000000.00,49692.00,14907600.00,2990000.00,14857908.00,99.67
1008042,4500000.00,72600000.00,22361400.00,360763920.00,68100000.00,338402520.00,93.80
1009615,4.00,10000.00,19.88,49692.00,9996.00,49672.12,99.96
1012642,25000.00,123.00,124230.00,611.21,-24877.00,-123618.79,-20225.20
1014127,500.00,10000.00,2484.60,49692.00,9500.00,47207.40,95.00
1014193,10.00,200.00,49.69,993.84,190.00,944.15,95.00
1024777,10000.00,60000.00,49692.00,298152.00,50000.00,248460.00,83.33
1026837,40000.00,800.00,198768.00,3975.36,-39200.00,-194792.64,-4900.00
1028194,1.00,400.00,4.97,1987.68,399.00,1982.71,99.75


In [0]:
df = spark.table(f"{catalogo}.bronze.tb_credits_and_tags")

# Domínio oficial de gêneros do TMDB (19 gêneros).
# Em vez de criar uma regra para cada tipo de lixo, só fica o que estiver nesta lista.
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama", "Family",
    "Fantasy", "History", "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "TV Movie", "Thriller", "War", "Western",
]

df = df.select(F.trim(F.col("id")).alias("id_filme"), F.col("genres"))

# Padroniza os separadores: ponto e vírgula e barra vertical viram vírgula
df = df.withColumn("genres", F.regexp_replace(F.col("genres"), "[;|]", ","))

# split quebra o texto numa lista; explode transforma cada item da lista em uma linha
df = df.withColumn("nome_genero", F.explode(F.split(F.col("genres"), ",")))

# Tira aspas e barras soltas e os espaços das pontas
df = df.withColumn("nome_genero", F.trim(F.regexp_replace(F.col("nome_genero"), r'["\\]', "")))

# Fica só o que for gênero válido. Remove vazios, números e textos deslocados.
df = df.filter(F.col("nome_genero").isin(generos_validos))

# O mesmo filme aparece mais de uma vez na origem, então tira as repetições
df = df.select("id_filme", "nome_genero").dropDuplicates()

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.silver.tb_generos")

print(df.count())
display(df.groupBy("nome_genero").count().orderBy(F.col("count").desc()))

142160


nome_genero,count
Drama,32683
Documentary,19248
Comedy,18861
Thriller,10412
Horror,9868
Romance,7723
Action,6114
Crime,4795
Animation,4529
TV Movie,4122


In [0]:
df = spark.table(f"{catalogo}.bronze.tb_credits_and_tags").withColumn("id_filme", F.trim(F.col("id")))

# 1) JUNTAR AS QUATRO COLUNAS EM UMA TABELA SÓ
# Para cada coluna: padroniza separadores (; e | viram vírgula), quebra em lista e explode em linhas.
# F.lit(tipo) cria uma coluna fixa com o tipo da entidade.
colunas = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista", "production_companies": "Produtora"}

partes = []
for coluna, tipo in colunas.items():
    parte = df.select(
        F.col("id_filme"),
        F.lit(tipo).alias("tipo_entidade"),
        F.explode(F.split(F.regexp_replace(F.col(coluna), "[;|]", ","), ",")).alias("nome"),
    )
    partes.append(parte)

entidades = partes[0].unionByName(partes[1]).unionByName(partes[2]).unionByName(partes[3])

# 2) LIMPEZA DE CADA NOME
# Tira aspas, colchetes, barras e espaços do começo e do fim
entidades = entidades.withColumn("nome", F.trim(F.regexp_replace(F.col("nome"), r'^["\\\[\] ]+|["\\\[\] ]+$', "")))

# Marcadores de "sem dado" que aparecem nessas colunas
ausencia = ["", "N/A", "[]", "NENHUM", "UNKNOWN", "NONE", "NÃO INFORMADO"]
entidades = entidades.filter(~F.upper(F.col("nome")).isin(ausencia))

# Se ainda sobrou aspa ou barra no meio, é pedaço de texto corrompido
entidades = entidades.filter(~F.col("nome").rlike(r'["\\]'))

# Precisa ter pelo menos uma letra (remove números deslocados como "0.6")
entidades = entidades.filter(F.col("nome").rlike(r"\p{L}"))

# Nome de pessoa e de empresa começa com maiúscula. O que começa com minúscula aqui são
# keywords deslocadas ("assassin", "midlife crisis") ou continuação de frase da sinopse.
entidades = entidades.filter(~F.col("nome").rlike(r"^\p{Ll}"))

# Textos longos são trechos de sinopse, não nomes
entidades = entidades.filter(F.length(F.col("nome")) <= 60)
entidades = entidades.filter(F.size(F.split(F.col("nome"), " ")) <= 8)

# 3) REMOVER GÊNEROS, PAÍSES E IDIOMAS QUE CAÍRAM NESSAS COLUNAS ("Drama", "English", "United States of America")
# Regra: se o valor aparece mais vezes nas colunas de tags (gêneros, países, idiomas)
# do que nas colunas de nomes, ele é tag e sai.
# "English" aparece milhares de vezes como idioma e poucas centenas como diretor: sai.
# "ARTE" (produtora) aparece mais de mil vezes como produtora e 27 como país: fica.
tags = df.select(
    F.explode(F.split(F.regexp_replace(F.concat_ws(",", "genres", "production_countries", "spoken_languages"), "[;|]", ","), ",")).alias("nome")
)
tags = tags.withColumn("chave", F.lower(F.trim(F.col("nome")))).groupBy("chave").agg(F.count("*").alias("vezes_tag"))

frequencia_nomes = entidades.withColumn("chave", F.lower(F.col("nome"))).groupBy("chave").agg(F.count("*").alias("vezes_nome"))

excluir = (
    tags.join(frequencia_nomes, "chave", "left")
    .filter(F.col("vezes_tag") > F.coalesce(F.col("vezes_nome"), F.lit(0)))
    .select("chave")
)

entidades = entidades.withColumn("chave", F.lower(F.col("nome")))
entidades = entidades.join(excluir, "chave", "left_anti")

# 4) PADRONIZAR CAPITALIZAÇÃO E REMOVER DUPLICADOS
entidades = entidades.withColumn("nome_entidade", F.initcap(F.col("nome")))
entidades = entidades.select("id_filme", "nome_entidade", "tipo_entidade").dropDuplicates()

entidades.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.silver.tb_pessoas_empresas")

print(entidades.count())
display(entidades.groupBy("tipo_entidade").count())

882243


tipo_entidade,count
Produtora,116215
Diretor,94508
Roteirista,125252
Ator,546268
